# Ontologies from Zero: Semantic Guardrails for Agentic Systems
Welcome to the hands-on session! In this notebook, you will build a dual-gate validation pipeline for agentic tool execution:
* **Gate 1**: Pydantic for structural validation.
* **Gate 2**: SHACL & RDFLib for semantic business guardrails.

In [ ]:
# Setup: Install required packages
!pip install pydantic rdflib pyshacl

## Step 1: Gate 1 - Structural Validation (Pydantic)
Define a Pydantic model to catch type errors and malformed inputs before they reach the agent or execution layer.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class ToolExecutionPayload(BaseModel):
    # TODO: Define fields for agent tool execution (e.g., agent_id, action_type, target_resource, amount)
    pass

# Test your payload
try:
    sample = ToolExecutionPayload(agent_id="agent_007", action_type="transfer", target_resource="account_42", amount=150.0)
    print("Gate 1 Passed:", sample.model_dump())
except Exception as e:
    print("Gate 1 Failed:", e)

from rdflib import Graph, Namespace, Literal, RDF, URIRef, XSD
Use RDFLib to construct the domain graph and pyshacl to validate business rules that static types cannot catch.

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, URIRef
from pyshacl import validate

# Define Namespaces
EX = Namespace("http://example.org/agent-ns#")
SH = Namespace("http://www.w3.org/ns/shacl#")

# TODO: Initialize an RDFLib Graph and add instance data representing a proposed agent action
data_graph = Graph()

print("Data graph initialized with", len(data_graph), "triples.")

In [ ]:
# TODO: Define SHACL shapes graph for business guardrails (e.g., transfers over a certain limit require manager approval)
shapes_graph = Graph()

# Run Validation
conforms, results_graph, results_text = validate(
    data_graph,
    shacl_graph=shapes_graph,
    inference='rdfs',
    abort_on_first=False,
    meta_shacl=False,
    advanced_features=True,
    js=False,
    serialization='turtle'
)

print("Gate 2 Conforms:", conforms)
print("Validation Report:\n", results_text)

## Step 3: Same Action, Now With Manager Approval
Give the action from Step 2 manager approval and re-run Gate 2. If your shapes graph encodes the rule correctly, the same over-limit transaction should now conform.

In [ ]:
# TODO: Update the proposed action in data_graph so it carries manager approval
#       (hint: Graph.set replaces an existing value for a subject/predicate pair)

conforms, results_graph, results_text = validate(
    data_graph,
    shacl_graph=shapes_graph,
    inference='rdfs',
    advanced_features=True,
)

print("Gate 2 Conforms (with approval):", conforms)